**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to Python

Python is the lingua franca of scientific computing — nearly every other workshop in this curriculum assumes it. By the end you will write real programs, organize them with functions and classes, and use NumPy + Matplotlib the way the [DSP](../../Intro_DSP/README.md) and [ML](../../Intro_Mach_Learn/README.md) workshops expect.

## 0. Introduction

Why Python? It reads almost like pseudocode, it has a library for everything, and — as we saw in [Intro to GPU Systems](../../Intro_GPU/README.md) — its scientific libraries hand the heavy lifting to fast compiled code underneath. You write the *what*, the libraries handle the *how fast*.

## 1. Pre-requisites

None! This is the entry point of the programming track.

To run this notebook you need a Python environment with Jupyter. The easiest paths:

- **Google Colab** (zero install): upload this notebook at [colab.research.google.com](https://colab.research.google.com/).
- **Local**: install [miniconda](https://docs.conda.io/en/latest/miniconda.html), then
  `conda create -n sps python=3.12 numpy matplotlib jupyter` and select the `sps` kernel.

If you can run the cell below, you're ready.

In [1]:
print("Hello, SPS!")

Hello, SPS!


---
### 🕐 Session 1 of 3 — *Language Core* (~35 min)
**Goal:** use variables, types, control flow, functions, and comprehensions.
**Feeds into:** Session 2 (data structures & objects).

---

## 2. The Language Core

### 2.1. Variables & Types

Python variables are *labels stuck onto objects* — you don't declare types, the object itself knows what it is. The core types: `int`, `float`, `str`, `bool`.

In [2]:
x = 42          # int
y = 3.14        # float
name = "Fourier"  # str
ready = True    # bool

print(type(x), type(y), type(name), type(ready))

<class 'int'> <class 'float'> <class 'str'> <class 'bool'>


Arithmetic works as expected, with two divisions worth memorizing: `/` always gives a float, `//` floors to an integer, and `%` gives the remainder. `**` is exponentiation.

In [3]:
print(7 / 2)    # 3.5   true division
print(7 // 2)   # 3     floor division
print(7 % 2)    # 1     remainder
print(2 ** 10)  # 1024  power

3.5
3
1
1024


### 2.2. Strings

Strings are sequences: you can index them, slice them, and glue them together. *f-strings* (the `f"..."` form) are the standard way to format output.

In [4]:
signal = "sinusoid"
print(signal[0], signal[-1])   # first and last character
print(signal[:4])              # slice: characters 0..3
print(f"A {signal} has {len(signal)} letters.")

s d
sinu
A sinusoid has 8 letters.


### 2.3. Control Flow

Python uses **indentation** instead of braces — the whitespace *is* the syntax. If you took [Intro to C](../Intro_C.ipynb), everything here will feel familiar, just lighter.

In [5]:
for k in range(5):          # k = 0, 1, 2, 3, 4
    if k % 2 == 0:
        print(k, "even")
    else:
        print(k, "odd")

0 even
1 odd
2 even
3 odd
4 even


In [6]:
# while loops: collatz steps from 27
steps, m = 0, 27
while m != 1:
    m = m // 2 if m % 2 == 0 else 3 * m + 1
    steps += 1
print(f"27 reaches 1 in {steps} steps")

27 reaches 1 in 111 steps


### 2.4. Functions

Same idea as in every language: name a computation, give it inputs, return outputs. Default arguments and returning multiple values are everyday Python.

In [7]:
def db(power, reference=1.0):
    """Convert a power ratio to decibels."""
    import math
    return 10 * math.log10(power / reference)

print(db(100))        # 20 dB
print(db(2, 1))       # ~3 dB — the famous "3 dB point" from filter design

20.0
3.010299956639812


💡 **Intuition.** A *comprehension* is a for-loop folded into a single expression: `[f(x) for x in xs if cond(x)]` reads as "the list of f(x), for each x, keeping only those passing the condition." It's the idiom you'll see everywhere in scientific Python.

In [8]:
squares = [k**2 for k in range(10)]
even_squares = [s for s in squares if s % 2 == 0]
print(squares)
print(even_squares)

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
[0, 4, 16, 36, 64]


---
### 🕐 Session 2 of 3 — *Data & Objects* (~35 min)
**Goal:** organize data with lists/dicts/sets; organize code with classes and modules.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (scientific Python).

---

## 3. Data Structures

### 3.1. Lists, Tuples, Dicts, Sets

Four containers cover nearly everything:

| Container | Mutable? | Best for |
|---|---|---|
| `list` | yes | ordered collections you'll grow/change |
| `tuple` | no | fixed groups (coordinates, returns) |
| `dict` | yes | lookup by key |
| `set` | yes | membership tests, de-duplication |

In [9]:
samples = [0.1, 0.5, 0.2, 0.5]        # list
point = (3.0, 4.0)                     # tuple
freqs = {"A4": 440.0, "C5": 523.25}    # dict
unique = set(samples)                  # set: removes the duplicate 0.5

samples.append(0.9)
print(samples)
print(freqs["A4"])
print(unique)

[0.1, 0.5, 0.2, 0.5, 0.9]
440.0
{0.1, 0.5, 0.2}


💡 **Intuition.** A `dict` is a *phone book*: you don't scan for an entry, you jump straight to it by name. Lookup cost doesn't grow with size — the same $O(1)$ magic as the hash tables discussed in [Intro to C](../Intro_C.ipynb).

### 3.2. Classes

A class bundles *data* and the *functions that operate on it*. You've already used them — everything in Python is an object. `__init__` runs at construction; `self` is the object being operated on.

In [10]:
class Signal:
    """A minimal sampled-signal container."""
    def __init__(self, samples, fs):
        self.samples = list(samples)
        self.fs = fs                     # sampling rate in Hz

    def duration(self):
        return len(self.samples) / self.fs

    def scaled(self, gain):
        return Signal([gain * s for s in self.samples], self.fs)

s = Signal([0.0, 0.7, 1.0, 0.7, 0.0], fs=100)
print(f"{s.duration():.2f} s long")
print(s.scaled(2.0).samples)

0.05 s long
[0.0, 1.4, 2.0, 1.4, 0.0]


### 3.3. Modules & Imports

Code is organized into *modules* (files) and *packages* (folders of files). `import` brings them in; the standard library is enormous.

In [11]:
import math
from collections import Counter

votes = ["fir", "iir", "fir", "fir", "iir"]
print(Counter(votes).most_common(1))
print(math.pi, math.e)

[('fir', 3)]
3.141592653589793 2.718281828459045


---
### 🕐 Session 3 of 3 — *Scientific Python* (~40 min)
**Goal:** compute with NumPy arrays and visualize with Matplotlib — the toolkit every later workshop assumes.
**Builds on:** Session 2. &nbsp; **Feeds into:** [DSP](../../Intro_DSP/README.md), [ML](../../Intro_Mach_Learn/README.md), [GPU](../../Intro_GPU/README.md) workshops.

---

## 4. NumPy

💡 **Intuition.** A Python list is a bag of arbitrary objects; a NumPy array is a *contiguous block of numbers of one type* — the same "shelf" picture as C arrays. That layout is what lets NumPy hand whole-array operations to fast vectorized machine code, as demonstrated in [Intro to GPU Systems](../../Intro_GPU/Intro_GPU.ipynb) §2.2.

In [12]:
import numpy as np

t = np.linspace(0, 1, 8)   # 8 points from 0 to 1
x = np.sin(2 * np.pi * t)  # operates on ALL elements at once — no loop
print(t.round(3))
print(x.round(3))

[0.    0.143 0.286 0.429 0.571 0.714 0.857 1.   ]
[ 0.     0.782  0.975  0.434 -0.434 -0.975 -0.782 -0.   ]


### 4.1. Vectorization

The golden rule: **if you're writing a `for` loop over a NumPy array, look for the array operation instead.** It's shorter *and* orders of magnitude faster.

In [13]:
import time

N = 1_000_000
data = np.random.default_rng(42).random(N)

tic = time.perf_counter()
out_loop = [2 * v + 1 for v in data]           # Python loop
t_loop = time.perf_counter() - tic

tic = time.perf_counter()
out_vec = 2 * data + 1                          # vectorized
t_vec = time.perf_counter() - tic

print(f"loop:       {t_loop:.4f} s")
print(f"vectorized: {t_vec:.4f} s  ({t_loop / t_vec:.0f}x faster)")
# exact timings vary by machine; the ratio is the point

loop:       0.0821 s
vectorized: 0.0042 s  (20x faster)


### 4.2. Shapes, Slicing & Broadcasting

In [14]:
A = np.arange(12).reshape(3, 4)   # 3 rows, 4 columns
print(A)
print(A[1, :])      # row 1
print(A[:, 2])      # column 2
print(A.T.shape)    # transpose: (4, 3)

# broadcasting: subtract each column's mean from the column
print((A - A.mean(axis=0)).round(2))

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[4 5 6 7]
[ 2  6 10]
(4, 3)
[[-4. -4. -4. -4.]
 [ 0.  0.  0.  0.]
 [ 4.  4.  4.  4.]]


💡 **Intuition.** *Broadcasting* stretches a smaller array across a bigger one without copying: subtracting a length-4 row of means from a (3, 4) matrix applies it to every row. If two shapes match from the right (or one side is 1), NumPy broadcasts.

## 5. Matplotlib

In [15]:
import matplotlib.pyplot as plt

fs = 500                                # sampling rate, Hz
t = np.arange(0, 1, 1 / fs)
clean = np.sin(2 * np.pi * 5 * t)       # 5 Hz sine
noisy = clean + 0.4 * np.random.default_rng(0).standard_normal(t.size)

plt.figure(figsize=(8, 3))
plt.plot(t, noisy, label="noisy", alpha=0.6)
plt.plot(t, clean, label="clean", linewidth=2)
plt.xlabel("time [s]"); plt.ylabel("amplitude")
plt.title("A 5 Hz sine in noise")
plt.legend(); plt.tight_layout(); plt.show()

/tmp/ipykernel_1833291/361073028.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.tight_layout(); plt.show()


One more — the plot you will make a hundred times in this curriculum: a spectrum. (A preview of [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb), where you'll learn what the FFT actually *is*.)

In [16]:
X = np.fft.rfft(noisy)
f = np.fft.rfftfreq(t.size, 1 / fs)

plt.figure(figsize=(8, 3))
plt.plot(f, np.abs(X) / t.size)
plt.xlim(0, 50)
plt.xlabel("frequency [Hz]"); plt.ylabel("|X(f)|")
plt.title("The 5 Hz peak pops right out of the noise")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1833291/3806610988.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Conclusion

You can now write Python programs, structure them with functions/classes, and lean on NumPy + Matplotlib for real numeric work. That's the exact toolkit the rest of the curriculum builds on.

---
## Where next

- [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) — that FFT peak, explained from first principles.
- [Intro to C](../Intro_C.ipynb) — what your Python objects look like one level down.
- [Intro to GPU Systems](../../Intro_GPU/README.md) — the same vectorization story, scaled to thousands of cores.
- [Intro to PyTorch](../../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb) — NumPy's deep-learning sibling.